Von MP4 zu Audio + Background
Original MP4
   ↓ Extract audio (keep original SR)
   ↓ Separate vocals/background (at original SR)
   ↓ Convert vocals & background stems → 48 kHz WAV
   ↓ Denoise vocals (48 kHz WAV) for Synthesis
   ↓ Transcripe text using WhisperX API
Audio + Background   



In [3]:
import os
import shutil
import shlex
import subprocess
import demucs.separate
from df.enhance import enhance, init_df, load_audio, save_audio
import json
import requests
import time


class WhisperXClient:
    def __init__(self, base_url="http://whisperx:5001"):
        self.base_url = base_url
        self._check_connection()
    
    def _check_connection(self, max_retries=5, retry_delay=10):
        """Check if WhisperX service is available with retries"""
        for attempt in range(max_retries):
            try:
                response = requests.get(f"{self.base_url}/health", timeout=10)
                if response.status_code == 200:
                    print(f"✅ WhisperX service connected: {response.json()}")
                    return
            except Exception as e:
                print(f"⚠️ Attempt {attempt + 1}/{max_retries}: WhisperX service not ready ({e})")
                if attempt < max_retries - 1:
                    print(f"   Retrying in {retry_delay} seconds...")
                    time.sleep(retry_delay)
                else:
                    print("❌ Failed to connect to WhisperX service. Make sure it's running.")
                    raise ConnectionError("WhisperX service unavailable")
    
    def transcribe(self, audio_path, language=None, return_timestamps=True, diarize=False, verbose=True):
        """
        Transcribe audio using WhisperX service
        
        Args:
            audio_path: Path to audio file
            language: Language code (optional, auto-detect if None)
            return_timestamps: Whether to return word-level timestamps
            diarize: Whether to perform speaker diarization
            verbose: Whether to print progress
        
        Returns:
            dict: Transcription result in Whisper-compatible format
        """
        if verbose:
            print(f"🎙️ Transcribing audio file: {audio_path}")
        
        try:
            with open(audio_path, 'rb') as audio_file:
                files = {'audio': audio_file}
                data = {
                    'language': language or '',
                    'return_timestamps': str(return_timestamps).lower(),
                    'diarize': str(diarize).lower()
                }
                
                if verbose:
                    print("   Sending request to WhisperX service...")
                
                response = requests.post(
                    f"{self.base_url}/transcribe",
                    files=files,
                    data=data,
                    timeout=300  # 5 minutes timeout
                )
                
                if response.status_code == 200:
                    result = response.json()
                    if result.get('success'):
                        # Convert WhisperX format to Whisper-compatible format
                        transcription = result['transcription']
                        
                        # Create Whisper-compatible result
                        whisper_result = {
                            'text': ' '.join([seg.get('text', '') for seg in transcription.get('segments', [])]),
                            'segments': transcription.get('segments', []),
                            'language': result.get('language', language or 'en')
                        }
                        
                        if verbose:
                            print(f"✅ Transcription complete. Found {len(whisper_result.get('segments', []))} segments.")
                            if whisper_result.get('segments'):
                                # Print segments like original Whisper
                                for segment in whisper_result['segments']:
                                    start = segment.get('start', 0)
                                    end = segment.get('end', 0)
                                    text = segment.get('text', '')
                                    print(f"[{start:05.3f} --> {end:05.3f}] {text}")
                        
                        return whisper_result
                    else:
                        raise Exception(f"WhisperX error: {result.get('error', 'Unknown error')}")
                else:
                    raise Exception(f"HTTP {response.status_code}: {response.text}")
                    
        except Exception as e:
            print(f"❌ Transcription failed: {str(e)}")
            raise e
    
    def health_check(self):
        """Check if WhisperX service is healthy"""
        try:
            response = requests.get(f"{self.base_url}/health", timeout=10)
            return response.json()
        except Exception as e:
            return {'error': str(e)}


def load_whisperx_client():
    """Initialize WhisperX client (replaces load_whisper_model)"""
    print("🔗 Connecting to WhisperX service...")
    client = WhisperXClient()
    print("✅ WhisperX client ready.")
    return client


def separate_vocals(input_audio_path, output_dir, model='mdx_extra'):
    """
    Separate vocals and no-vocals with Demucs and save to custom output_dir.
    """
    print(f"Separating vocals from {input_audio_path}...")

    # Run Demucs CLI separation
    cmd = f'--two-stems vocals -n {model} "{input_audio_path}"'
    demucs.separate.main(shlex.split(cmd))

    base_name = os.path.splitext(os.path.basename(input_audio_path))[0]
    demucs_output_dir = os.path.join("separated", model, base_name)

    vocals_src = os.path.join(demucs_output_dir, "vocals.wav")
    no_vocals_src = os.path.join(demucs_output_dir, "no_vocals.wav")

    os.makedirs(output_dir, exist_ok=True)

    vocals_dst = os.path.join(output_dir, f"{base_name}_vocals.wav")
    no_vocals_dst = os.path.join(output_dir, f"{base_name}_no_vocals.wav")

    shutil.copy2(vocals_src, vocals_dst)
    shutil.copy2(no_vocals_src, no_vocals_dst)

    print(f"Vocals saved to: {vocals_dst}")
    print(f"No vocals saved to: {no_vocals_dst}")

    return vocals_dst, no_vocals_dst  # Return paths of saved files


def convert_to_48k_wav(input_audio_path, output_wav_path):
    """
    Convert any audio input to 48kHz stereo WAV using ffmpeg.
    """
    print(f"Converting {input_audio_path} to 48kHz WAV at {output_wav_path}...")
    subprocess.run([
        'ffmpeg', '-y', '-i', input_audio_path,
        '-ar', '48000',
        '-ac', '2',
        output_wav_path
    ], check=True)
    print("Conversion done.")


def denoise_audio(input_wav_path, output_wav_path):
    """
    Denoise/enhance the WAV audio using DeepFilterNet.
    """
    print(f"Denoising audio {input_wav_path} to {output_wav_path}...")
    model, df_state, _ = init_df()
    audio, _ = load_audio(input_wav_path, sr=df_state.sr())
    enhanced = enhance(model, df_state, audio)
    save_audio(output_wav_path, enhanced, df_state.sr())
    print("Denoising done.")


def separate_audio_from_video(video_path, audio_out_path, muted_video_out_path):
    """Extract audio from video and save both audio and muted video separately."""
    
    # Ensure output directories exist
    os.makedirs(os.path.dirname(audio_out_path), exist_ok=True)
    os.makedirs(os.path.dirname(muted_video_out_path), exist_ok=True)
    
    # Extract audio (48 kHz mono WAV)
    audio_cmd = [
        "ffmpeg", "-y", "-i", video_path,
        "-vn", "-acodec", "pcm_s16le", "-ar", "48000", "-ac", "1",
        audio_out_path
    ]

    # Remove audio from video
    muted_video_cmd = [
        "ffmpeg", "-y", "-i", video_path,
        "-an", muted_video_out_path
    ]

    print("🔊 Extracting audio...")
    subprocess.run(audio_cmd, check=True)
    print(f"✅ Audio saved to: {audio_out_path}")

    print("🎥 Saving muted video...")
    subprocess.run(muted_video_cmd, check=True)
    print(f"✅ Muted video saved to: {muted_video_out_path}")


def transcribe_audio(whisperx_client, audio_file_path, language=None, diarize=False, **kwargs):
    """Transcribe audio using WhisperX client and return full result including timestamps."""
    
    # Just pass all the parameters directly - the client will handle them
    return whisperx_client.transcribe(
        audio_file_path, 
        language=language, 
        return_timestamps=True,
        diarize=diarize,
        verbose=True,
        **kwargs  # Pass any additional parameters
    )
    
def write_transcription_to_json(result, output_path):
    """Write the full WhisperX transcription result to a JSON file."""
    os.makedirs(os.path.dirname(output_path), exist_ok=True)  # Create output dir
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(result, f, indent=2, ensure_ascii=False)
    print(f"✅ JSON transcript saved to: {output_path}")


I use a function to make sure all audio, video, etc. are in one folder. To increase structure of the repo

In [4]:
def generate_pipeline_paths(video_path, base_output="../output"):
    video_title = os.path.splitext(os.path.basename(video_path))[0]
    root_dir = os.path.join(base_output, video_title)

    return {
        "root": root_dir,
        "audio_raw": os.path.join(root_dir, "audio", "original_unprocessed.wav"),
        "audio_vocals": os.path.join(root_dir, "audio", "vocals_raw.wav"),
        "audio_vocals_48k": os.path.join(root_dir, "audio", "vocals_48k.wav"),
        "audio_vocals_denoised": os.path.join(root_dir, "audio", "vocals_denoised.wav"),
        "video_muted": os.path.join(root_dir, "video", "video_muted.mp4"),
        "video_final": os.path.join(root_dir, "video", "video_final.mp4"),
        "transcript_json": os.path.join(root_dir, "transcripts", "transcript.json"),
        "demucs_out": os.path.join(root_dir, "demucs")
    }


this is the main function to use the entire pipeline 

Von MP4 zu Audio + Background + Transcriped text with WhisperX
   

In [5]:
def full_video_audio_pipeline(video_path, base_output="../output", diarize=False, force_preprocessing=False):
    from df.enhance import enhance, init_df, load_audio, save_audio

    paths = generate_pipeline_paths(video_path, base_output)
    os.makedirs(paths["root"], exist_ok=True)

    # Check if preprocessing files already exist
    preprocessing_complete = (
        os.path.exists(paths["audio_raw"]) and
        os.path.exists(paths["audio_vocals"]) and
        os.path.exists(paths["audio_vocals_48k"]) and
        os.path.exists(paths["audio_vocals_denoised"])
    )

    if force_preprocessing or not preprocessing_complete:
        print("Running preprocessing steps...")
        # Step 1: Separate audio and muted video
        separate_audio_from_video(video_path, paths["audio_raw"], paths["video_muted"])

        # Step 2: Separate vocals with Demucs
        vocals_path, _ = separate_vocals(paths["audio_raw"], output_dir=os.path.dirname(paths["audio_vocals"]))
        shutil.move(vocals_path, paths["audio_vocals"])

        # Step 3: Convert vocals to 48kHz WAV
        convert_to_48k_wav(paths["audio_vocals"], paths["audio_vocals_48k"])

        # Step 4: Denoise vocals
        model_df, df_state, _ = init_df()
        audio, _ = load_audio(paths["audio_vocals_48k"], sr=df_state.sr())
        enhanced = enhance(model_df, df_state, audio)
        save_audio(paths["audio_vocals_denoised"], enhanced, df_state.sr())
    else:
        print("Preprocessing files found, skipping to transcription...")

    # Step 5: Transcribe with WhisperX
    whisperx_client = load_whisperx_client()
    result = transcribe_audio(whisperx_client, paths["audio_raw"], diarize=diarize)
    write_transcription_to_json(result, paths["transcript_json"])

    print(f"🎉 Pipeline complete. All outputs saved under: {paths['root']}")
    return paths


Here is how to use this pipeline:

In [7]:
# Test the pipeline
video_path = "../input/job.mp4"
pipeline_paths = full_video_audio_pipeline(video_path, diarize=True)  # Set diarize=True for speaker identification


Preprocessing files found, skipping to transcription...
🔗 Connecting to WhisperX service...
✅ WhisperX service connected: {'compute_type': 'float16', 'cuda_available': True, 'device': 'cuda', 'model_status': 'loadable', 'status': 'healthy', 'torch_version': '2.1.0+cu121', 'whisperx_version': 'unknown'}
✅ WhisperX client ready.
🎙️ Transcribing audio file: ../output/job/audio/original_unprocessed.wav
   Sending request to WhisperX service...
❌ Transcription failed: HTTP 500: {"error":"Transcription error: All model loading strategies failed. Last error: HTTP Error 301: Moved Permanently"}



Exception: HTTP 500: {"error":"Transcription error: All model loading strategies failed. Last error: HTTP Error 301: Moved Permanently"}


## Transcription Comparison Function (Updated for WhisperX)

In [ ]:
import difflib
import os

def compare_transcriptions(whisperx_client, paths, language=None, max_chars=500, diarize=False):
    """
    Compare transcriptions from raw, vocals, and denoised audio using WhisperX.
    
    Args:
        whisperx_client: WhisperX client instance
        paths: Dictionary from generate_pipeline_paths() with all file paths
        language: Optional language code for transcription
        max_chars: Number of characters to preview in each transcript
        diarize: Whether to perform speaker diarization
    """
    
    print("🔍 Comparing transcriptions from different audio versions...")

    # Transcribe all versions
    print("\n📝 Transcribing raw audio...")
    result_raw = transcribe_audio(whisperx_client, paths["audio_raw"], language, diarize)
    
    print("\n📝 Transcribing vocals-only audio...")
    result_vocals = transcribe_audio(whisperx_client, paths["audio_vocals"], language, diarize)
    
    print("\n📝 Transcribing denoised vocals...")
    result_denoised = transcribe_audio(whisperx_client, paths["audio_vocals_denoised"], language, diarize)

    # Extract text
    text_raw = result_raw["text"].strip()
    text_vocals = result_vocals["text"].strip()
    text_denoised = result_denoised["text"].strip()

    # Print first few lines of each
    print("\n" + "="*60)
    print("📊 TRANSCRIPTION COMPARISON RESULTS")
    print("="*60)
    print("\n--- RAW AUDIO ---")
    print(text_raw[:max_chars] + ("..." if len(text_raw) > max_chars else ""))
    print("\n--- VOCALS ONLY ---")
    print(text_vocals[:max_chars] + ("..." if len(text_vocals) > max_chars else ""))
    print("\n--- DENOISED VOCALS ---")
    print(text_denoised[:max_chars] + ("..." if len(text_denoised) > max_chars else ""))

    # Compare similarities
    sim_raw_vs_vocals = difflib.SequenceMatcher(None, text_raw, text_vocals).ratio()
    sim_raw_vs_denoised = difflib.SequenceMatcher(None, text_raw, text_denoised).ratio()
    sim_vocals_vs_denoised = difflib.SequenceMatcher(None, text_vocals, text_denoised).ratio()

    print("\n🔗 Similarity Scores:")
    print(f"Raw vs Vocals:      {sim_raw_vs_vocals:.2%}")
    print(f"Raw vs Denoised:    {sim_raw_vs_denoised:.2%}")
    print(f"Vocals vs Denoised: {sim_vocals_vs_denoised:.2%}")

    # Analyze which version performed best
    print("\n📈 Analysis:")
    if sim_raw_vs_vocals > 0.95:
        print("✅ Vocal separation preserved transcription quality well")
    else:
        print("⚠️ Vocal separation may have affected transcription quality")
    
    if sim_vocals_vs_denoised > 0.95:
        print("✅ Denoising preserved transcription quality well")
    else:
        print("⚠️ Denoising may have affected transcription quality")

    # Save all transcripts
    out_dir = os.path.join(paths["root"], "transcripts")
    os.makedirs(out_dir, exist_ok=True)
    write_transcription_to_json(result_raw, os.path.join(out_dir, "raw.json"))
    write_transcription_to_json(result_vocals, os.path.join(out_dir, "vocals.json"))
    write_transcription_to_json(result_denoised, os.path.join(out_dir, "denoised.json"))

    print("\n📝 All transcripts saved for manual review.")
    print("="*60)
    
    return {
        'raw': result_raw,
        'vocals': result_vocals,
        'denoised': result_denoised,
        'similarities': {
            'raw_vs_vocals': sim_raw_vs_vocals,
            'raw_vs_denoised': sim_raw_vs_denoised,
            'vocals_vs_denoised': sim_vocals_vs_denoised
        }
    }


## Run Comparison Test

In [ ]:
# Run transcription comparison
whisperx_client = load_whisperx_client()
paths = generate_pipeline_paths(video_path)

# Only run if files exist (from previous pipeline run)
if os.path.exists(paths["audio_raw"]):
    comparison_results = compare_transcriptions(
        whisperx_client, 
        paths, 
        diarize=False  # Set to True if you want speaker identification
    )
else:
    print("⚠️ Please run the full pipeline first to generate audio files.")


## Advanced WhisperX Features Demo

In [ ]:
def demo_whisperx_features(audio_path):
    """Demo advanced WhisperX features like diarization and multiple languages"""
    client = load_whisperx_client()
    
    print("🎭 Testing WhisperX Speaker Diarization...")
    result_with_speakers = client.transcribe(
        audio_path, 
        diarize=True, 
        return_timestamps=True
    )
    
    print("\n👥 Speakers identified:")
    speakers = set()
    for segment in result_with_speakers.get('segments', []):
        speaker = segment.get('speaker', 'Unknown')
        speakers.add(speaker)
        print(f"[{segment.get('start', 0):.1f}s] Speaker {speaker}: {segment.get('text', '')}")
    
    print(f"\n📊 Total speakers detected: {len(speakers)}")
    
    return result_with_speakers

# Uncomment to test speaker diarization
# if os.path.exists("../input/job.mp4"):
#     paths = generate_pipeline_paths("../input/job.mp4")
#     if os.path.exists(paths["audio_raw"]):
#         demo_whisperx_features(paths["audio_raw"])


## Health Check and Troubleshooting

In [ ]:
def check_services_health():
    """Check health of all services"""
    print("🏥 Checking service health...")
    
    # Check WhisperX
    try:
        client = WhisperXClient()
        health = client.health_check()
        print(f"✅ WhisperX: {health}")
    except Exception as e:
        print(f"❌ WhisperX: {e}")
    
    # Check Fish Speech (if accessible)
    try:
        response = requests.get("http://fish-speech:7860/health", timeout=5)
        print(f"✅ Fish Speech: {response.status_code}")
    except:
        print("⚠️ Fish Speech: Not accessible from this container")
    
    print("Health check complete.")

# Run health check
check_services_health()
